In [1]:
import os
os.makedirs("models", exist_ok=True)
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.initializers import GlorotUniform, HeNormal, RandomNormal, GlorotNormal, HeUniform
import tensorflow as tf
import random

SEED = 72
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

train_df = pd.read_csv('california_housing_train.csv')
test_df = pd.read_csv('california_housing_test.csv')

features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
target = 'median_house_value'

x_train_full = train_df[features].values
y_train_full = train_df[target].values
x_test = test_df[features].values
y_test = test_df[target].values


mean = x_train_full.mean(axis=0)
std = x_train_full.std(axis=0)
x_train = (x_train_full - mean) / std
x_test = (x_test - mean) / std

2025-11-01 19:45:30.302021: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-01 19:45:30.302727: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-01 19:45:30.385263: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-01 19:45:32.051493: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
def build_model(
    layers_config,
    activations,
    initializers,  # <-- добавлено
    optimizer_name,
    learning_rate,
    dropout_rates,
    use_batch_norm,
    kernel_regularizer,
    bias_regularizer
):
    model = Sequential()
    for i, (units, act, init) in enumerate(zip(layers_config, activations, initializers)):
        if i == 0:
            model.add(Dense(units,
                            activation=act,
                            input_shape=(x_train.shape[1],),
                            kernel_initializer=init,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        else:
            model.add(Dense(units,
                            activation=act,
                            kernel_initializer=init,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        if use_batch_norm[i]:
            model.add(BatchNormalization())
        if dropout_rates[i] > 0:
            model.add(Dropout(dropout_rates[i]))
    # Выходной слой
    model.add(Dense(1, activation='linear', kernel_initializer=GlorotUniform(seed=SEED)))
    # ... (остальное без изменений)
    if optimizer_name == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        opt = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("Unsupported optimizer")
    model.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return model

In [3]:
import itertools

def random_architecture():
    n_layers = np.random.choice([2, 3, 4])
    start = np.random.choice([8, 16, 32, 64]) 
    layers = [start]
    for _ in range(1, n_layers):
        next_units = np.random.choice([u for u in [8, 16, 32, 64] if u <= layers[-1]])
        layers.append(next_units)
    return layers

def random_activation(n):
    acts = []
    for _ in range(n):
        acts.append(np.random.choice(['relu', 'elu','tanh']))
    return acts

def random_dropout(n):
    return [np.random.choice([0.0, 0.1, 0.2]) for _ in range(n)]

def random_batch_norm(n):
    return [np.random.choice([True, False]) for _ in range(n)]

def random_regularizer():
    choice = np.random.choice(['none', 'l1', 'l2'])
    if choice == 'l1':
        return l1(1e-4)
    elif choice == 'l2':
        return l2(1e-4)
    else:
        return None
# надо связать с функцией активации 
def get_initializer_for_activation(activation):
    if activation in ('relu', 'elu'):
        # He инициализация
        choice = np.random.choice(['henorm', 'he'])
        if choice == 'henorm':
            return HeNormal(seed=SEED)
        else:
            return HeUniform(seed=SEED)
    elif activation == 'tanh':
        # Glorot инициализация
        choice = np.random.choice(['glorotnorm', 'glorot'])
        if choice == 'glorotnorm':
            return GlorotNormal(seed=SEED)
        else:
            return GlorotUniform(seed=SEED)
    else:
        # fallback
        return GlorotUniform(seed=SEED)

def random_optimizer():
    #return np.random.choice(['adam', 'sgd', 'rmsprop'])
    return np.random.choice(['adam','rmsprop','adamw'])

def random_lr():
    return np.random.choice([1e-4, 5e-4, 1e-3, 5e-3, 1e-2])

def random_batch_size():
    return np.random.choice([16, 32, 64, 128])

def random_epochs():
    return np.random.choice([200,300,400])

In [ ]:
import gc
from tensorflow.keras import backend as K
results = []
N_EXPERIMENTS = 500
INTERVAL = 25 

early_stop = EarlyStopping(monitor='val_mae', patience=150, restore_best_weights=True)

for exp_id in range(N_EXPERIMENTS):
    print(f"Эксперимент {exp_id + 1}/{N_EXPERIMENTS}")
    
    # Генерация
    layers = random_architecture()
    n = len(layers)
    activations = random_activation(n)

    initializers = [get_initializer_for_activation(act) for act in activations]
    
    dropout_rates = random_dropout(n)
    batch_norm_flags = random_batch_norm(n)
    kernel_reg = random_regularizer()
    bias_reg = random_regularizer()
    opt_name = random_optimizer()
    lr = random_lr()
    batch_size = random_batch_size()
    max_epochs = random_epochs()
    

    try:
        model = build_model(
            layers_config=layers,
            activations=activations,
            optimizer_name=opt_name,
            learning_rate=lr,
            dropout_rates=dropout_rates,
            use_batch_norm=batch_norm_flags,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            initializers=initializers
        )
    except Exception as e:
        print(f"Ошибка при создании модели: {e}")
        continue

    history = model.fit(
        x_train_full, y_train_full,
        validation_split=0.2,
        epochs=max_epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    
    actual_epochs = len(history.history['mae'])
    
    train_mae = history.history['mae'][-1]
    val_mae = history.history['val_mae'][-1]
    test_mae = model.evaluate(x_test, y_test, verbose=1)[1]

    model_save_path = f"models/model_{exp_id + 1}.keras"
    model.save(model_save_path)
    
    # Сохранение результата
    results.append({
        'ID': exp_id + 1,
        'layers': str(layers),
        'activations': str(activations),
        'optimizer': opt_name,
        'learning_rate': lr,
        'batch_size': batch_size,
        'epochs': actual_epochs,
        'dropout': str(dropout_rates),
        'batch_norm': str(batch_norm_flags),
        'kernel_regularizer': str(kernel_reg),
        'bias_regularizer': str(bias_reg),
        'initializer': str([init.__class__.__name__ for init in initializers]),
        'train_mae': train_mae,
        'val_mae': val_mae,
        'test_mae': test_mae
    })
    
    del model
    K.clear_session()
    gc.collect()
    # Промежуточная запись каждые INTERVAL экспериментов
    if (exp_id + 1) % INTERVAL == 0:
        df_partial = pd.DataFrame(results)
        df_partial.to_csv("experiment_results2.csv", index=False)
        print(f"Промежуточные результаты (до эксперимента {exp_id + 1}) сохранены в experiment_results.csv")

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('experiment_results2.csv', on_bad_lines='skip')

df['test_mae'] = pd.to_numeric(df['test_mae'], errors='coerce')

valid = df[np.isfinite(df['test_mae'])]

top_10 = valid.nsmallest(10, 'test_mae')

result = top_10[['ID', 'test_mae', 'layers', 'activations', 'optimizer', 'learning_rate', 'batch_size', 'epochs', 'dropout', 'batch_norm']]

result

,ID,test_mae,layers,activations,optimizer,learning_rate,batch_size,epochs,dropout,batch_norm
3,4,62006.843750,"[np.int64(32), np.int64(8), np.int64(8), np.in...","[np.str_('elu'), np.str_('relu'), np.str_('tan...",rmsprop,0.010,64,400,"[np.float64(0.1), np.float64(0.1), np.float64(...","[np.False_, np.True_, np.True_, np.False_]"
180,268,85975.289062,"[np.int64(16), np.int64(8), np.int64(8), np.in...","[np.str_('relu'), np.str_('elu'), np.str_('rel...",rmsprop,0.005,16,150,"[np.float64(0.0), np.float64(0.1), np.float64(...","[np.True_, np.False_, np.False_, np.False_]"
5,6,101339.250000,"[np.int64(64), np.int64(32), np.int64(16)]","[np.str_('relu'), np.str_('tanh'), np.str_('el...",rmsprop,0.010,16,280,"[np.float64(0.0), np.float64(0.0), np.float64(...","[np.True_, np.False_, np.True_]"
34,52,102161.148438,"[np.int64(32), np.int64(32), np.int64(16)]","[np.str_('elu'), np.str_('relu'), np.str_('tan...",adam,0.005,32,200,"[np.float64(0.0), np.float64(0.0), np.float64(...","[np.True_, np.True_, np.True_]"
217,325,118258.617188,"[np.int64(16), np.int64(16)]","[np.str_('relu'), np.str_('elu')]",adam,0.010,16,150,"[np.float64(0.2), np.float64(0.0)]","[np.True_, np.False_]"
136,206,127601.289062,"[np.int64(64), np.int64(32), np.int64(16)]","[np.str_('elu'), np.str_('tanh'), np.str_('elu')]",adam,0.005,16,150,"[np.float64(0.2), np.float64(0.1), np.float64(...","[np.True_, np.True_, np.False_]"
232,347,130147.437500,"[np.int64(16), np.int64(16)]","[np.str_('relu'), np.str_('elu')]",adam,0.005,16,150,"[np.float64(0.0), np.float64(0.1)]","[np.True_, np.False_]"
134,204,155887.781250,"[np.int64(8), np.int64(8), np.int64(8)]","[np.str_('relu'), np.str_('relu'), np.str_('re...",adam,0.010,64,150,"[np.float64(0.2), np.float64(0.2), np.float64(...","[np.True_, np.False_, np.False_]"
83,124,155948.656250,"[np.int64(64), np.int64(32), np.int64(32)]","[np.str_('tanh'), np.str_('tanh'), np.str_('re...",adam,0.010,16,150,"[np.float64(0.0), np.float64(0.1), np.float64(...","[np.True_, np.False_, np.False_]"
204,306,156029.468750,"[np.int64(64), np.int64(16), np.int64(16)]","[np.str_('relu'), np.str_('elu'), np.str_('rel...",adam,0.010,16,150,"[np.float64(0.0), np.float64(0.0), np.float64(...","[np.True_, np.True_, np.False_]"
